# Gemma 4 Socratic tutor (E4B or E2B): 4-bit baseline vs LoRA fine-tuning (Colab T4)

Model: `google/gemma-4-E4B-it` by default (7.9B parameters including per-layer embeddings, about 4B
effective), or `google/gemma-4-E2B-it` (5.1B, about 2B effective) with `MODEL_SIZE = "E2B"`. Both load in
4-bit NF4, the lowest precision bitsandbytes supports for LoRA training. The per-layer embedding tables stay
in 16-bit, so E4B needs about 9.5 GB for weights and E2B about 6.8 GB on the T4's 15 GB. Only text is used;
LoRA is attached to the language model's attention and MLP layers, never to the vision or audio parts.

One 4-bit baseline and three LoRA settings, all scored on the same 100 evaluation contexts
(50 SocraTeach, 50 MathDial) and the same 100 validation dialogues. Each LoRA setting trains on
the same subset of the candidate train set and is saved as its own fine-tuned model.

Expected time on a Colab T4: E2B roughly an hour, E4B roughly twice that. Neither has been timed on a
T4 yet; the first LoRA setting prints `train_minutes`, so check it and lower `TRAIN_DIALOGUES` if needed.
Results, adapters and a resume cache are written to Google Drive, so a disconnect does not lose
finished settings: reconnect and run all cells again.

Before running: Runtime > Change runtime type > T4 GPU. Put `candidate_train.jsonl(.gz)` and
`candidate_val.jsonl(.gz)` from the data-prep notebook in your Drive or upload them to `/content`.

| Metric | Meaning | Better |
|---|---|---|
| leak rate | reply states the final answer before the student has said it | lower |
| question rate | reply contains a question | higher for a Socratic tutor |
| ROUGE-L, BLEU | word overlap with the real tutor's next turn | higher |
| words | reply length (real tutors: about 24) | closer to real tutors |
| val loss | masked loss on the target tutor turns of the validation dialogues | lower |

In [ ]:
SMOKE_TEST = False     # True: tiny random model on CPU, a few examples (checks the code, not the results)

import os, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
IN_COLAB = "google.colab" in sys.modules
if not SMOKE_TEST:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                           "transformers>=5.0,<6", "peft", "accelerate", "bitsandbytes", "rouge_score", "sacrebleu"])
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import gc, glob, gzip, json, random, re, time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from rouge_score import rouge_scorer
import sacrebleu
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_SIZE = "E4B"        # "E4B" (about 4B effective, ~9.5 GB in 4-bit) or "E2B" (about 2B effective, ~6.8 GB)
MODEL_ID, REVISION = {"E4B": ("google/gemma-4-E4B-it", "ee0ef6023621cff504d758262d4e04895a5af4a2"),
                      "E2B": ("google/gemma-4-E2B-it", "3e22461f65e89153144f8adb70e3b8c2cc9845a7")}[MODEL_SIZE]   # Apache 2.0, not gated
MAX_LEN = 1024            # tokens per training dialogue; longer ones are cut at a tutor turn
TRAIN_DIALOGUES = 300     # training subset shared by every LoRA setting; lower it if the first setting is too slow
EVAL_PER_SOURCE = 50      # evaluation contexts per source (SocraTeach, MathDial)
VAL_PER_SOURCE = 50       # validation dialogues per source for the loss
MAX_NEW_TOKENS = 96
GEN_BATCH = 4 if MODEL_SIZE == "E4B" else 8   # smaller generation batches leave room for E4B on a 15 GB T4
SEED = 42

# LoRA settings: Unsloth's Gemma 4 default (r=8, alpha=8), a larger rank, and a lower learning rate
GRID = [dict(r=8, lora_alpha=8, learning_rate=2e-4),
        dict(r=16, lora_alpha=16, learning_rate=2e-4),
        dict(r=8, lora_alpha=8, learning_rate=5e-5)]
# Language-model projections only; the pattern excludes the vision and audio towers
TARGET_MODULES = r"^(?!.*(vision|audio)).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
COMMON = dict(lora_dropout=0.05, batch_size=1, grad_accum=8, warmup=0.05, epochs=1, target_modules=TARGET_MODULES)

if SMOKE_TEST:
    MAX_LEN, TRAIN_DIALOGUES, EVAL_PER_SOURCE, VAL_PER_SOURCE, MAX_NEW_TOKENS, GEN_BATCH = 512, 8, 2, 2, 8, 2
    GRID = GRID[:2]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if Path("/content/drive/MyDrive").exists():
    OUT = Path(f"/content/drive/MyDrive/gemma4_{MODEL_SIZE.lower()}_socratic")
elif Path("/content").exists():
    OUT = Path(f"/content/gemma4_{MODEL_SIZE.lower()}_socratic")
else:
    OUT = Path("fast_output")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| output", OUT)

CACHE_FILE = OUT / "run_cache.json"   # finished models; a rerun skips them
CACHE = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
def save_cache():
    CACHE_FILE.write_text(json.dumps(CACHE, indent=1, default=str))

RESULTS = {"config": dict(model_size=MODEL_SIZE, model=MODEL_ID, revision=REVISION, max_len=MAX_LEN, train_dialogues=TRAIN_DIALOGUES,
                          eval_per_source=EVAL_PER_SOURCE, val_per_source=VAL_PER_SOURCE, max_new_tokens=MAX_NEW_TOKENS,
                          grid=GRID, common=COMMON, smoke_test=SMOKE_TEST)}

## 1. Data

In [ ]:
def find_file(name):
    # .jsonl or .jsonl.gz, uploaded to /content, in Drive, or next to the notebook
    for pattern in [f"/content/{name}.jsonl*", f"/content/**/{name}.jsonl*", f"/content/drive/MyDrive/**/{name}.jsonl*",
                    f"../socratic_data_prep/processed/{name}.jsonl*", f"processed/{name}.jsonl*"]:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(f"{name}.jsonl(.gz) not found: upload it to /content or put it in your Drive")

def read_jsonl(path):
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

TRAIN_FILE, VAL_FILE = find_file("candidate_train"), find_file("candidate_val")
train_rows, val_rows = read_jsonl(TRAIN_FILE), read_jsonl(VAL_FILE)

# candidate_val lists all MathDial dialogues first, so evaluation and validation sets are drawn per source
def per_source(rows, n, seed):
    rng = random.Random(seed)
    out = []
    for src in ["socrateach", "mathdial"]:
        pool = [r for r in rows if r["source"] == src]
        out += rng.sample(pool, min(n, len(pool)))
    return out

eval_rows = per_source(val_rows, EVAL_PER_SOURCE, SEED)
loss_rows = per_source(val_rows, VAL_PER_SOURCE, SEED + 1)
print(TRAIN_FILE, "| train", Counter(r["source"] for r in train_rows),
      "| eval", Counter(r["source"] for r in eval_rows), "| val loss", Counter(r["source"] for r in loss_rows))

## 2. Prompt format and training targets

Each dialogue starts with a system prompt holding the tutoring instruction, the problem, the correct
final answer (for checking, never to reveal) and, for MathDial, the student's first attempt. Tutor
turns are the model role and student turns the user role. Tutor turns are training targets unless
MathDial labels them "telling" or they reveal the answer before the student has said it; those turns
stay as context without loss.

The text is rendered with Gemma's own chat template (thinking off). Each tutor turn's content and its
end-of-turn marker are located in that text, and the loss mask is set from those character offsets.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
tok.pad_token = tok.pad_token or tok.eos_token

SYSTEM = ("You are a Socratic maths tutor. Guide the student one step at a time with questions and hints. "
          "Never state the final answer.\n\nProblem: {question}\n"
          "Correct final answer (for checking only, never reveal it): {answer}")

def chat_messages(rec, upto=None):
    sys_msg = SYSTEM.format(question=rec["question"], answer=rec["reference_answer"])
    if rec.get("student_attempt"):
        sys_msg += "\n\nStudent's first attempt:\n" + rec["student_attempt"]
    msgs = rec["messages"] if upto is None else rec["messages"][:upto]
    return [{"role": "system", "content": sys_msg}] + [
        {"role": "assistant" if m["role"] == "tutor" else "user", "content": m["content"]} for m in msgs]

def template(msgs, generation_prompt=False):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=generation_prompt, enable_thinking=False)

# The end-of-turn marker that follows an assistant message (for Gemma 4: "<turn|>")
_probe = template([{"role": "user", "content": "U"}, {"role": "assistant", "content": "QQQ"}])
END_OF_TURN = _probe[_probe.index("QQQ") + 3:].split("\n")[0]
print("end-of-turn marker:", repr(END_OF_TURN))

def render(rec, n_messages):
    # Official template text plus the character span (content + end marker) of each tutor turn
    msgs = chat_messages(rec, n_messages)
    text, spans, pos = template(msgs), [], 0
    for i, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        start = text.find(m["content"] + END_OF_TURN, pos)
        if start < 0:
            return None, None
        end = start + len(m["content"]) + len(END_OF_TURN)
        src = rec["messages"][i - 1]
        spans.append((start, end, src.get("move") != "telling" and not src.get("leaks_answer", False)))
        pos = end
    return text, spans

found = sum(render(r, len(r["messages"]))[0] is not None for r in train_rows[:200])
assert found == min(200, len(train_rows)), f"tutor turns not found in the template text for {min(200, len(train_rows)) - found} dialogues"
print("tutor-turn spans located in the chat template text")

def encode(rec):
    n = len(rec["messages"])
    while n > 0:
        if rec["messages"][n - 1]["role"] == "tutor":
            text, spans = render(rec, n)
            if text is None:
                return None
            enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
            if len(enc["input_ids"]) <= MAX_LEN:
                break
        n -= 1
    if n == 0:
        return None
    labels = [-100] * len(enc["input_ids"])
    for k, (a, b) in enumerate(enc["offset_mapping"]):
        if any(train and a >= s and b <= e for s, e, train in spans):
            labels[k] = enc["input_ids"][k]
    return None if all(l == -100 for l in labels) else dict(input_ids=enc["input_ids"], labels=labels)

train_all = [e for e in map(encode, train_rows) if e]
train_enc = random.Random(SEED).sample(train_all, min(TRAIN_DIALOGUES, len(train_all)))
val_enc = [e for e in map(encode, loss_rows) if e]
RESULTS["data"] = dict(train_dialogues=len(train_enc), train_tokens=sum(len(e["input_ids"]) for e in train_enc),
                       val_dialogues=len(val_enc))
print(RESULTS["data"])

## 3. Evaluation contexts and metrics

For each evaluation dialogue, one tutor turn that follows a student message is chosen. The model
sees everything before it and writes the next tutor message; the real tutor's message is the reference.

In [ ]:
NUM = re.compile(r"(?<![\d./])-?\d[\d,]*(?:\.\d+)?(?!\d|\.\d|/)")

def numbers(text):
    return {n.replace(",", "").rstrip("0").rstrip(".") if "." in n else n.replace(",", "") for n in NUM.findall(text)}

def answer_key(rec):
    found = numbers(rec["reference_answer"])
    key = found.pop() if len(found) == 1 else None
    return None if key in numbers(rec["question"]) else key

rng = random.Random(SEED)
eval_points = []
for rec in eval_rows:
    idx = [i for i, m in enumerate(rec["messages"]) if m["role"] == "tutor" and i > 0 and rec["messages"][i - 1]["role"] == "student"]
    if not idx:
        continue
    i = rng.choice(idx)
    key = answer_key(rec)
    said = key is not None and any(key in numbers(m["content"]) for m in rec["messages"][:i] if m["role"] == "student")
    eval_points.append(dict(dialogue_id=rec["dialogue_id"], source=rec["source"], key=key, student_said=said,
                            prompt=template(chat_messages(rec, i), generation_prompt=True),
                            reference=rec["messages"][i]["content"]))
print(len(eval_points), "evaluation contexts", Counter(p["source"] for p in eval_points))

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score(replies, points):
    df = pd.DataFrame([dict(leak=p["key"] is not None and not p["student_said"] and p["key"] in numbers(r),
                            question="?" in r, words=len(r.split()), rougeL=scorer.score(p["reference"], r)["rougeL"].fmeasure)
                       for r, p in zip(replies, points)])
    out = dict(leak_rate=df.leak.mean(), question_rate=df.question.mean(), words=df.words.mean(), rougeL=df.rougeL.mean(),
               bleu=sacrebleu.corpus_bleu(replies, [[p["reference"] for p in points]]).score / 100)
    return {k: round(float(v), 4) for k, v in out.items()}

RESULTS["real_tutors"] = {**score([p["reference"] for p in eval_points], eval_points), "rougeL": 1.0, "bleu": 1.0}
print("real tutors:", RESULTS["real_tutors"])

## 4. Model helpers

In [ ]:
def load_model():
    if SMOKE_TEST:   # random 4-layer Gemma 4 text model with the real vocabulary
        from transformers import Gemma4TextConfig, Gemma4ForCausalLM
        cfg = Gemma4TextConfig(vocab_size=len(tok), vocab_size_per_layer_input=len(tok), hidden_size=64, intermediate_size=128,
                               num_hidden_layers=4, layer_types=["sliding_attention", "full_attention"] * 2,
                               num_attention_heads=4, num_key_value_heads=1, head_dim=16, global_head_dim=16,
                               hidden_size_per_layer_input=8, num_kv_shared_layers=2, sliding_window=64,
                               pad_token_id=tok.pad_token_id, bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id)
        return Gemma4ForCausalLM(cfg).to(DEVICE)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_use_double_quant=True,
                               llm_int8_skip_modules=["lm_head", "per_layer_input_gate", "per_layer_projection", "per_layer_model_projection"])
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float16, device_map={"": 0},
                                                 quantization_config=quant)
    # The T4 has no bf16, and Gemma can overflow in fp16: stop early instead of training on inf/nan
    with torch.no_grad():
        e = val_enc[0]
        loss = model(input_ids=torch.tensor([e["input_ids"]], device=DEVICE), labels=torch.tensor([e["labels"]], device=DEVICE)).loss
    assert torch.isfinite(loss), "fp16 forward pass gave inf/nan on the T4; this model cannot be trained in fp16 here"
    print("fp16 sanity check loss:", round(float(loss), 3))
    return model

def free():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, points):
    model.eval()
    tok.padding_side = "left"
    replies = []
    for i in range(0, len(points), GEN_BATCH):
        batch = tok([p["prompt"] for p in points[i:i + GEN_BATCH]], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        out = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
        for seq in out[:, batch["input_ids"].shape[1]:]:
            replies.append(tok.decode(seq, skip_special_tokens=True).strip())
    tok.padding_side = "right"
    return replies

@torch.no_grad()
def val_loss(model):
    model.eval()
    total, count = 0.0, 0
    for e in val_enc:
        ids, labels = torch.tensor([e["input_ids"]], device=DEVICE), torch.tensor([e["labels"]], device=DEVICE)
        n = int((labels[:, 1:] != -100).sum())
        total += float(model(input_ids=ids, labels=labels).loss) * n
        count += n
    return round(total / count, 4)

def evaluate(model, name):
    t = time.time()
    replies = generate(model, eval_points)
    metrics = {**score(replies, eval_points), "val_loss": val_loss(model), "eval_minutes": round((time.time() - t) / 60, 2)}
    if any(not r for r in replies):
        metrics["empty_replies"] = sum(not r for r in replies)
    pd.DataFrame(dict(dialogue_id=[p["dialogue_id"] for p in eval_points], source=[p["source"] for p in eval_points],
                      reference=[p["reference"] for p in eval_points], reply=replies)).to_csv(OUT / f"generations_{name}.csv", index=False)
    print(name, metrics)
    return metrics

## 5. Baseline: pretrained Gemma 4 in 4-bit, no training

In [ ]:
BASE = load_model()
if "baseline_4bit" not in CACHE:
    CACHE["baseline_4bit"] = evaluate(BASE, "baseline_4bit"); save_cache()
RESULTS["baseline_4bit"] = CACHE["baseline_4bit"]
print("baseline_4bit", RESULTS["baseline_4bit"])

## 6. LoRA fine-tuning with three hyperparameter settings

Every setting trains for one epoch on the same `TRAIN_DIALOGUES` dialogues (effective batch 8,
cosine schedule, 5% warm-up, 8-bit paged AdamW, gradient checkpointing, dropout 0.05) and is scored
on the same contexts as the baseline. The 4-bit model is loaded once; each setting gets a fresh
adapter, which is saved and then removed before the next setting.

In [ ]:
if not SMOKE_TEST:
    BASE = prepare_model_for_kbit_training(BASE, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})

def collate(batch):
    width = max(len(e["input_ids"]) for e in batch)
    pad = lambda seq, v: seq + [v] * (width - len(seq))
    return dict(input_ids=torch.tensor([pad(e["input_ids"], tok.pad_token_id) for e in batch]),
                labels=torch.tensor([pad(e["labels"], -100) for e in batch]),
                attention_mask=torch.tensor([pad([1] * len(e["input_ids"]), 0) for e in batch]))

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def release_adapter(model):
    base = model.unload()
    if hasattr(base, "peft_config"):
        del base.peft_config
    assert not any("lora_" in n for n, _ in base.named_modules()), "LoRA layers left on the base model"
    return base

for setting in GRID:
    name = f"lora_r{setting['r']}_lr{setting['learning_rate']:g}"
    if name in CACHE:
        print(name, "(already done)", CACHE[name]); continue
    torch.manual_seed(SEED)
    model = get_peft_model(BASE, LoraConfig(r=setting["r"], lora_alpha=setting["lora_alpha"], lora_dropout=COMMON["lora_dropout"],
                                            target_modules=COMMON["target_modules"], bias="none", task_type="CAUSAL_LM"))
    args = TrainingArguments(output_dir=str(OUT / "checkpoints" / name), num_train_epochs=COMMON["epochs"],
                             per_device_train_batch_size=COMMON["batch_size"], gradient_accumulation_steps=COMMON["grad_accum"],
                             learning_rate=setting["learning_rate"], lr_scheduler_type="cosine", warmup_steps=COMMON["warmup"],
                             logging_steps=1 if SMOKE_TEST else 5, save_strategy="no", report_to="none", seed=SEED,
                             fp16=DEVICE == "cuda", optim="paged_adamw_8bit" if DEVICE == "cuda" else "adamw_torch",
                             gradient_checkpointing=DEVICE == "cuda", gradient_checkpointing_kwargs={"use_reentrant": False},
                             remove_unused_columns=False,
                             max_steps=2 if SMOKE_TEST else -1)
    t = time.time()
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_enc), data_collator=collate)
    trainer.train()
    history = [dict(step=h["step"], loss=h["loss"]) for h in trainer.state.log_history if "loss" in h]
    model = trainer.model
    trainer.accelerator.free_memory(); del trainer; free()
    train_minutes = round((time.time() - t) / 60, 2)
    metrics = evaluate(model, name)
    model.save_pretrained(OUT / "adapters" / name)
    CACHE[name] = dict(**setting, **metrics, train_minutes=train_minutes, history=history,
                       trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad))
    save_cache()
    BASE = release_adapter(model); del model; free()
    if DEVICE == "cuda":
        print(f"GPU memory allocated after {name}: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

## 7. Results

Selection rule for the recommended setting: lowest validation loss among the settings whose leak rate
is no higher than the baseline's; if none qualifies, the lowest leak rate.

In [ ]:
names = [f"lora_r{s['r']}_lr{s['learning_rate']:g}" for s in GRID]
tuned = {n: CACHE[n] for n in names}
RESULTS["tuned"] = {n: {k: v for k, v in m.items() if k != "history"} for n, m in tuned.items()}
ok = [n for n in names if tuned[n]["leak_rate"] <= RESULTS["baseline_4bit"]["leak_rate"]]
best = min(ok, key=lambda n: tuned[n]["val_loss"]) if ok else min(names, key=lambda n: (tuned[n]["leak_rate"], tuned[n]["val_loss"]))
RESULTS["recommended"] = best

cols = ["leak_rate", "question_rate", "words", "rougeL", "bleu", "val_loss"]
table = pd.DataFrame({"real tutors": RESULTS["real_tutors"], "baseline 4-bit": RESULTS["baseline_4bit"],
                      **{n: tuned[n] for n in names}}).T[cols]
display(table)
print("recommended setting:", best)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for n in names:
    ax[0].plot([h["step"] for h in tuned[n]["history"]], [h["loss"] for h in tuned[n]["history"]], label=n)
ax[0].set(title="Training loss", xlabel="step", ylabel="loss"); ax[0].legend(fontsize=8)
models = ["baseline 4-bit"] + names
ax[1].bar(models, [RESULTS["baseline_4bit"]["val_loss"]] + [tuned[n]["val_loss"] for n in names], color="#4C72B0")
ax[1].set(title="Validation loss (lower is better)"); ax[1].tick_params(axis="x", rotation=25)
x = np.arange(3)
for k, (label, row) in enumerate([("real tutors", RESULTS["real_tutors"]), ("baseline 4-bit", RESULTS["baseline_4bit"])] +
                                  [(n, tuned[n]) for n in names]):
    ax[2].bar(x + (k - 2) * 0.16, [row["leak_rate"], row["question_rate"], row["rougeL"]], 0.16, label=label)
ax[2].set_xticks(x, ["leak rate", "question rate", "ROUGE-L"]); ax[2].set(title="Reply metrics on the same 100 contexts")
ax[2].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figures/baseline_vs_lora.png"); plt.show()

side = pd.read_csv(OUT / "generations_baseline_4bit.csv").merge(
    pd.read_csv(OUT / f"generations_{best}.csv")[["dialogue_id", "reply"]], on="dialogue_id", suffixes=("_baseline", "_tuned"))
display(side.head(5))
json.dump(RESULTS, open(OUT / "results.json", "w"), indent=1, default=str)
print("saved to", OUT)

## Notes

- Each fine-tuned model sees 300 dialogues for one epoch (about 38 optimizer steps). That is enough to compare settings quickly, not a full training run.
- Unsloth's Gemma 4 guide notes that fp16 on a T4 can overflow (reported for audio inputs). The notebook checks the first forward pass and stops if the loss is not finite.
- Gemma 4 E2B and E4B share key/value states between their last layers (20 of 35 and 18 of 42). Gradient checkpointing runs in non-reentrant mode so gradients still reach the layers that produce those states.
- Evaluation uses 100 contexts, so a difference of one or two leaks (1–2 points) is within noise.
- The leak check covers single-number final answers that are not already in the problem. Question rate counts question marks, a rough proxy for Socratic style.
- ROUGE-L and BLEU compare with one real tutor turn; a good Socratic reply can differ from it.
- The MathDial test split and the 161 team scenarios are not used; they stay for the final evaluation.